<a href="https://colab.research.google.com/github/deboraqlopes/desafio_busca-semantica-ibre/blob/main/D%C3%A9bora_Lopes_Desafio_t%C3%A9cnico_FGV_IBRE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Desafio Técnico**
## Estágio em Ciência de Dados

## **Etapa 1** - Limpeza e tratamento dos dados

In [ ]:
!pip install sentence-transformers beautifulsoup4 -q

In [ ]:
import requests
import json

url = "https://raw.githubusercontent.com/MateusRestier/desafio-ciencia-de-dados-ibre/main/dados/noticias_brutas.json"
noticias = json.loads(requests.get(url).text)

print(f"Total de notícias: {len(noticias)}")
print("\nExemplo bruto:")
print(noticias[0]['texto'][:300])

Total de notícias: 20

Exemplo bruto:
<p>Publicado em: 02/08/2023 às 20h15</p>


<p>O <strong>Comitê de Política Monetária</strong> (Copom) do <strong>Banco Central do Brasil</strong> decidiu, por unanimidade, manter a taxa básica de juros (Selic) em 13,75% ao ano. Esta é a quarta reunião consecutiva em que o comitê opta pela manutenção


In [ ]:
import re
from bs4 import BeautifulSoup

def limpar_texto(texto):
    # Remover tags HTML e entidades
    soup = BeautifulSoup(texto, "html.parser")
    texto = soup.get_text(separator=" ")

    # Remover datas de publicação vazadas no corpo do texto
    texto = re.sub(r'[Pp]ublicado\s+em:?\s*[\d/]+\s*(às|as)?\s*[\dh]+\s*', '', texto)
    texto = re.sub(r'\d{2}/\d{2}/\d{4}\s*[-—]?\s*[\dh:]*\s*\|?\s*[A-Za-zÀ-ÿ\s]*?(?=\s{2,}|\s[A-ZÁÉÍÓÚÀÂÊÔÃÕÇ])', '', texto)

    # Remover outros padrões de metadados comuns ex: "[tags: economia]"
    texto = re.sub(r'\[.*?\]', '', texto)

    # Remover múltiplas quebras de linha
    texto = re.sub(r'\n+', ' ', texto)

    # Remover espaços múltiplos
    texto = re.sub(r' +', ' ', texto)

    # Remover espaços nas bordas
    texto = texto.strip()

    return texto

In [ ]:
# Aplicar a limpeza em todas as notícias
dados_limpos = []
for noticia in noticias:
    dados_limpos.append({
        "id": noticia["id"],
        "titulo": noticia["titulo"],
        "texto": limpar_texto(noticia["texto"]),
        "data": noticia["data"],
        "fonte": noticia["fonte"]
    })

# Comparar antes e depois para verificar
print("=== ANTES ===")
print(noticias[0]['texto'][:300])
print("\n=== DEPOIS ===")
print(dados_limpos[0]['texto'][:300])

=== ANTES ===
<p>Publicado em: 02/08/2023 às 20h15</p>


<p>O <strong>Comitê de Política Monetária</strong> (Copom) do <strong>Banco Central do Brasil</strong> decidiu, por unanimidade, manter a taxa básica de juros (Selic) em 13,75% ao ano. Esta é a quarta reunião consecutiva em que o comitê opta pela manutenção

=== DEPOIS ===
O Comitê de Política Monetária (Copom) do Banco Central do Brasil decidiu, por unanimidade, manter a taxa básica de juros (Selic) em 13,75% ao ano. Esta é a quarta reunião consecutiva em que o comitê opta pela manutenção. Segundo o comunicado oficial, o Copom avaliou que o ambiente inflacionário ain


In [ ]:
print("Tamanho dos textos limpos:\n")
for n in dados_limpos:
    tamanho = len(n['texto'])
    alerta = " ⚠️ CONTEÚDO MÍNIMO" if tamanho < 100 else ""
    print(f"ID {n['id']:2d} | {tamanho:4d} chars | {n['titulo'][:50]}{alerta}")

Tamanho dos textos limpos:

ID  1 |  567 chars | Copom mantém Selic em 13,75% ao ano pela quarta re
ID  2 |  478 chars | IPCA de julho registra 0,12%, menor resultado para
ID  3 |  564 chars | PIB do Brasil cresce 1,9% no segundo trimestre de 
ID  4 |  563 chars | Taxa de desemprego cai para 7,9% no segundo trimes
ID  5 |  576 chars | Dólar fecha abaixo de R$ 4,80 com expectativa de c
ID  6 |  523 chars | Copom inicia ciclo de corte e reduz Selic para 13,
ID  7 |  620 chars | Inadimplência das famílias sobe para 6,3% em julho
ID  8 |  555 chars | Exportações brasileiras somam US$ 32,4 bilhões em 
ID  9 |  586 chars | Inflação ao produtor (IPA) desacelera e pressão so
ID 10 |  616 chars | Crédito total no Brasil atinge R$ 5,6 trilhões com
ID 11 |  613 chars | Selic deve recuar a 9% até o fim de 2024, projetam
ID 12 |  614 chars | PIB do agronegócio cresce 15,1% no primeiro semest
ID 13 |  444 chars | Balança comercial fecha julho com superávit de R$ 
ID 14 |  635 chars | Desemprego juve

In [ ]:
print("=== ANTES (bruto) ===")
print(repr(noticias[17]['texto']))

print("\n=== DEPOIS (limpo) ===")
print(repr(dados_limpos[17]['texto']))

=== ANTES (bruto) ===
'   Selic.'

=== DEPOIS (limpo) ===
'Selic.'


In [ ]:
# Marcando como inválido textos com menos de 100 caracteres
for noticia in dados_limpos:
    noticia['valido'] = len(noticia['texto']) >= 100

# Conferindo
for n in dados_limpos:
    if not n['valido']:
        print(f"ID {n['id']} marcada como inválida: '{n['texto']}'")

ID 18 marcada como inválida: 'Selic.'


In [ ]:
padroes = {
    "timestamp dd/mm/aaaa": r'\d{2}/\d{2}/\d{4}',
    "tag HTML restante":    r'<[^>]+>',
    "entidade HTML":        r'&[a-zA-Z]+;',
    "metadado embutido":    r'\[.*?\]',
    "espaços múltiplos":    r' {2,}',
}

print("Verificando resíduos em todos os textos limpos:\n")
encontrou_algo = False

for noticia in dados_limpos:
    if not noticia['valido']:
        continue
    for nome, padrao in padroes.items():
        matches = re.findall(padrao, noticia['texto'])
        if matches:
            encontrou_algo = True
            print(f"ID {noticia['id']:2d} | {nome}: {matches}")

if not encontrou_algo:
    print("Nenhum resíduo encontrado!")

Verificando resíduos em todos os textos limpos:

Nenhum resíduo encontrado!


In [ ]:
# Salvando o arquivo
with open('dados_limpos.json', 'w', encoding='utf-8') as f:
    json.dump(dados_limpos, f, ensure_ascii=False, indent=2)

print("\ndados_limpos.json salvo com sucesso!")
print(f"Total: {len(dados_limpos)} notícias ({sum(1 for n in dados_limpos if n['valido'])} válidas, {sum(1 for n in dados_limpos if not n['valido'])} inválidas)")


dados_limpos.json salvo com sucesso!
Total: 20 notícias (19 válidas, 1 inválidas)


##**Etapa 2**- Geração de embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Modelo carregado")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo carregado


In [ ]:
import numpy as np

# Selecionando as notícias válidas
noticias_validas = [n for n in dados_limpos if n['valido']]

# Combinando título + texto para embeddings mais ricos
corpus = [f"{n['titulo']}. {n['texto']}" for n in noticias_validas]

# Gerando os embeddings
embeddings = modelo.encode(corpus, show_progress_bar=True)

print(f"\nEmbeddings gerados!")
print(f"Shape: {embeddings.shape}  →  {len(noticias_validas)} notícias x {embeddings.shape[1]} dimensões")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embeddings gerados!
Shape: (19, 384)  →  19 notícias x 384 dimensões


In [ ]:
np.save('embeddings.npy', embeddings)

# Salvando também o mapeamento id -> posição no array
mapeamento = [{"posicao": i, "id": n["id"], "titulo": n["titulo"]}
              for i, n in enumerate(noticias_validas)]

with open('mapeamento.json', 'w', encoding='utf-8') as f:
    json.dump(mapeamento, f, ensure_ascii=False, indent=2)

print("embeddings.npy salvo!")
print("mapeamento.json salvo!")

embeddings.npy salvo!
mapeamento.json salvo!


## **Etapa 3**- Motor de Busca Semântico

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def buscar(query, top_k=3):
    # Transformar a query em embedding
    embedding_query = modelo.encode([query])

    # Calcular similaridade com todas as notícias
    similaridades = cosine_similarity(embedding_query, embeddings)[0]

    # Ordenar do mais similar para o menos
    indices_ordenados = np.argsort(similaridades)[::-1][:top_k]

    # Retornar os resultados
    print(f"\nQuery: '{query}'")
    print("=" * 60)
    for i, idx in enumerate(indices_ordenados):
        noticia = noticias_validas[idx]
        score = similaridades[idx]
        print(f"\n#{i+1} | Score: {score:.4f}")
        print(f"Título: {noticia['titulo']}")
        print(f"Texto: {noticia['texto'][:150]}...")

In [ ]:
queries = [
    "mudanças na taxa de juros",
    "mercado de trabalho e desemprego",
    "inflação e preços ao consumidor"
]

for query in queries:
    buscar(query)
    print()


Query: 'mudanças na taxa de juros'

#1 | Score: 0.6206
Título: Copom mantém Selic em 13,75% ao ano pela quarta reunião consecutiva
Texto: O Comitê de Política Monetária (Copom) do Banco Central do Brasil decidiu, por unanimidade, manter a taxa básica de juros (Selic) em 13,75% ao ano. Es...

#2 | Score: 0.6053
Título: Crédito total no Brasil atinge R$ 5,6 trilhões com desaceleração no crescimento
Texto: Imprensa O estoque total de crédito no sistema financeiro nacional atingiu R$ 5,6 trilhões em julho de 2023, segundo o Relatório de Crédito do Banco C...

#3 | Score: 0.5976
Título: Selic deve recuar a 9% até o fim de 2024, projetam economistas
Texto: Economistas consultados pelo Banco Central no Relatório Focus revisaram para baixo a projeção para a taxa Selic ao final de 2024, de 9,25% para 9,00% ...


Query: 'mercado de trabalho e desemprego'

#1 | Score: 0.6545
Título: Desemprego juvenil no Brasil ainda preocupa apesar de melhora geral
Texto: Apesar da melhora generalizada no merca